# 🏠 Bonus — Mehrere Merkmale

## Ein Haus ist mehr als seine Fläche

**Optional.** Für alle, die mit dem Hauptnotebook durch sind.

Dein bisheriges Modell kennt genau eine Zahl über ein Haus: die Wohnfläche. Damit erklärt es rund
die Hälfte der Preisunterschiede — R² ≈ 0,49, gemessen in Abschnitt 8 des Hauptnotebooks. Hier nimmst du zwei weitere Merkmale dazu und
schaust, wie weit du kommst.

**Voraussetzung:** das Hauptnotebook. Alles von dort — `mse()`, `rmse()`, `bewerte()`
und das Modell mit einem Merkmal — bekommst du hier geschenkt.

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |
| 💬 | Diskussionsfrage |

Es sind 2 Challenges.

---
## 0 · Setup

▶️ Alles aus dem Hauptnotebook auf einen Schlag — inklusive des Modells mit einem Merkmal, gegen
das wir uns gleich messen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})
np.set_printoptions(suppress=True, precision=2)

print("Setup fertig ✔")

In [ ]:
def lade_daten(dateiname):
    """Liest eine der Workshop-CSVs ein — lokal oder in Google Colab."""
    kandidaten = [
        Path(dateiname),
        Path("data") / dateiname,
        Path("..") / "data" / dateiname,
        Path("challenges/lineare-regression/data") / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            print(f"Gelesen: {pfad}")
            return pd.read_csv(pfad)

    # Google Colab: Datei von Hand hochladen
    try:
        from google.colab import files
        print(f"'{dateiname}' nicht gefunden — bitte jetzt hochladen:")
        files.upload()
        return pd.read_csv(dateiname)
    except ImportError:
        raise FileNotFoundError(
            f"'{dateiname}' nicht gefunden. Lege die CSV neben dieses Notebook."
        )


from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

df = lade_daten("haus_preise_einfach.csv")

# Dieselbe Aufteilung wie im Hauptnotebook
zufall = np.random.default_rng(42)
gemischt = zufall.permutation(len(df))
grenze = int(0.8 * len(df))

train, test = df.iloc[gemischt[:grenze]], df.iloc[gemischt[grenze:]]
x_train = train["wohnflaeche_qm"].to_numpy()
y_train = train["preis_usd"].to_numpy().astype(float)
x_test = test["wohnflaeche_qm"].to_numpy()
y_test = test["preis_usd"].to_numpy().astype(float)


def mse(y_wahr, y_vorhersage):
    """Aus dem Hauptnotebook: mittlerer quadratischer Fehler"""
    return float(np.mean((y_vorhersage - y_wahr) ** 2))


def rmse(y_wahr, y_vorhersage):
    """Aus dem Hauptnotebook: Wurzel daraus — in USD"""
    return float(np.sqrt(mse(y_wahr, y_vorhersage)))


def bewerte(name, y_wahr, y_vorhersage):
    """Aus dem Hauptnotebook: MAE, RMSE und R² auf einen Blick"""
    mae = float(np.mean(np.abs(y_vorhersage - y_wahr)))
    r2 = r2_score(y_wahr, y_vorhersage)
    print(f"{name:<28} MAE {mae:>10,.0f} USD   RMSE {rmse(y_wahr, y_vorhersage):>10,.0f} USD   R² {r2:>6.3f}")
    return r2


# Das Modell aus dem Hauptnotebook — unsere Messlatte
modell = LinearRegression().fit(x_train.reshape(-1, 1), y_train)

r2_baseline = bewerte("Baseline (Mittelwert)", y_test, np.full(len(y_test), y_train.mean()))
r2_einfach = bewerte("1 Merkmal (Fläche)", y_test, modell.predict(x_test.reshape(-1, 1)))

---
## 1 · Die Idee

📖 Bisher: eine Zahl rein, eine Zahl raus.

$$\text{preis} = w \cdot \text{wohnflaeche} + b$$

Jetzt geben wir dem Modell mehr Informationen. Jedes Merkmal bekommt sein **eigenes Gewicht**,
und alle Beiträge werden addiert:

$$\text{preis} = w_1 \cdot \text{wohnflaeche} + w_2 \cdot \text{qualitaet}
+ w_3 \cdot \text{baujahr} + b$$

Anschaulich: Aus der Geraden im Diagramm wird eine **Ebene im Raum** — sie kann sich in mehrere
Richtungen gleichzeitig neigen.

Und das ist exakt die Formel von der Folie *Scores* aus der Präsentation:

$$\text{score} = w_1 x_1 + w_2 x_2 + \dots + w_{784} x_{784} + b$$

Dort waren es 784 Pixel eines Bildes, hier drei Hausmerkmale. Das Prinzip ist identisch — und
das ist der Grund, warum sich diese eine Formel durch das ganze Machine Learning zieht.

**Für `scikit-learn` ändert sich am Code fast nichts:** statt einer Spalte übergeben wir drei.

---
## 2 · Die Daten

📖 Dieselben 21.613 Häuser, nur mit zwei zusätzlichen Spalten:

| Merkmal | Bedeutung |
|---|---|
| `wohnflaeche_qm` | Wohnfläche in m² |
| `qualitaet` | Bauqualität und Ausstattung, Index von 1 (einfach) bis 13 (luxuriös), vergeben von der Bauaufsicht des King County |
| `baujahr` | Jahr der Fertigstellung |

In [ ]:
df_multi = lade_daten("haus_preise_multi.csv")
df_multi.head()

In [ ]:
MERKMALE = ["wohnflaeche_qm", "qualitaet", "baujahr"]

# Exakt dieselbe Aufteilung wie im Hauptnotebook (gleiche Zufallszahl, gleiche Zeilen)
train_multi = df_multi.iloc[gemischt[:grenze]]
test_multi = df_multi.iloc[gemischt[grenze:]]

X_train = train_multi[MERKMALE].to_numpy(dtype=float)
X_test = test_multi[MERKMALE].to_numpy(dtype=float)

print(f"X_train hat die Form {X_train.shape}  (Häuser × Merkmale)")
print("Die ersten drei Trainingshäuser:")
print(X_train[:3])

📖 Groß geschriebenes `X` ist Konvention: ein großes `X` steht für eine **Tabelle** (Matrix),
ein kleines `x` für eine einzelne Zahlenreihe (Vektor). Deshalb brauchen wir hier auch kein
`.reshape(-1, 1)` mehr — `X_train` ist bereits eine Tabelle mit drei Spalten.

### 🛠️ Bonus-Challenge 1 — Dein Modell mit drei Merkmalen

In drei Schritten. Alles, was du brauchst, kennst du aus dem Hauptnotebook — nur mit drei Spalten
statt einer.

1. Ein `LinearRegression`-Modell erzeugen und mit `X_train`, `y_train` trainieren.
2. Die Preise für die Testdaten vorhersagen (`.predict(...)`).
3. Den Preis für ein einzelnes Haus schätzen: **140 m², Qualität 8, Baujahr 1995**.

*Tipp zu Schritt 3: `.predict()` erwartet eine Liste von Häusern, also
`modell_multi.predict([[140.0, 8.0, 1995.0]])[0]`.*

In [ ]:
# TODO 1: Modell erzeugen und trainieren
modell_multi = ...


# TODO 2: Preise für die Testdaten vorhersagen
y_pred_multi = ...

# TODO 3: Preis für ein Haus mit 140 m², Qualität 8, Baujahr 1995
mein_haus_preis = ...

print(f"Geschätzter Preis für 140 m², Qualität 8, Baujahr 1995: {mein_haus_preis:,.0f} USD")

In [ ]:
# ✅ Selbsttest
assert len(y_pred_multi) == len(y_test), "Für jedes Testhaus eine Vorhersage"
assert abs(mein_haus_preis - 400_729) < 2000, \
    f"Erwartet werden rund 400.700 USD, dein Modell sagt {mein_haus_preis:,.0f}"
assert abs(r2_score(y_test, y_pred_multi) - 0.586) < 0.01, "R² weicht ab — richtige Merkmale benutzt?"
print("✅ Bonus-Challenge 1 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
modell_multi = LinearRegression()
modell_multi.fit(X_train, y_train)

y_pred_multi = modell_multi.predict(X_test)

mein_haus_preis = modell_multi.predict([[140.0, 8.0, 1995.0]])[0]

print(f"Geschätzter Preis für 140 m², Qualität 8, Baujahr 1995: {mein_haus_preis:,.0f} USD")
```

Bis auf `X_train` statt `x_train.reshape(-1, 1)` ist das Zeile für Zeile dasselbe wie im
Hauptnotebook. Genau darum geht es: Ob ein Merkmal oder drei — am Code ändert sich nichts.

</details>

▶️ **Hat sich der Aufwand gelohnt?**

In [ ]:
print("Ergebnis auf den TESTDATEN")
print("-" * 78)
bewerte("Baseline (Mittelwert)", y_test, np.full(len(y_test), y_train.mean()))
bewerte("1 Merkmal (Fläche)", y_test, modell.predict(x_test.reshape(-1, 1)))
r2_multi = bewerte("3 Merkmale", y_test, y_pred_multi)

In [ ]:
namen = ["Baseline\n(Mittelwert)", "1 Merkmal\n(Fläche)", "3 Merkmale\n(+ Qualität, Baujahr)"]
werte = [max(r2_baseline, 0), r2_einfach, r2_multi]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
balken = ax.bar(namen, werte, color=[GRAU, BLAU, TEAL], width=0.55)
for balken_einzeln, wert in zip(balken, werte):
    ax.text(balken_einzeln.get_x() + balken_einzeln.get_width() / 2, wert + 0.015,
            f"{wert:.3f}", ha="center", fontweight="bold")

ax.set_ylabel("R² auf den Testdaten")
ax.set_ylim(0, 0.72)
ax.set_title("Mehr relevante Merkmale, bessere Vorhersage")
ax.grid(axis="x", visible=False)
plt.show()

📖 Der typische Fehler sinkt von **171.530** auf **148.976 USD** — rund 22.500 USD besser, allein
durch zwei zusätzliche Spalten, die schon die ganze Zeit in den Daten standen.

---
## 3 · Was hat das Modell gelernt?

📖 Ein lineares Modell hat einen großen Vorteil gegenüber einem neuronalen Netz: Man kann direkt
nachlesen, was es gelernt hat. Jedes Gewicht sagt, wie stark ein Merkmal den Preis beeinflusst.

In [ ]:
einheiten = ["pro m² Wohnfläche", "pro Qualitätsstufe", "pro Baujahr"]

print("Preisänderung laut Modell, wenn ein Merkmal um 1 steigt")
print("(und alle anderen gleich bleiben):")
print()
for merkmal, gewicht, einheit in zip(MERKMALE, modell_multi.coef_, einheiten):
    print(f"  {gewicht:>+12,.0f} USD  {einheit}")

💬 **Der Koeffizient für `baujahr` ist negativ — neuere Häuser sind laut Modell billiger?
Und die Wohnfläche ist plötzlich weniger wert als vorher (1.881 statt 3.000 USD/m²).**

<details>
<summary>Antwort aufklappen</summary>

Beides sind wichtige Lektionen über die Interpretation von Modellen:

**1. "Alles andere bleibt gleich" ist der Schlüssel.** Der Koeffizient sagt: Von zwei Häusern
mit *gleicher Fläche und gleicher Bauqualität* ist das ältere teurer. In Seattle stimmt das —
die alten Häuser stehen in den zentralen, teuren Vierteln, die Neubauten am Stadtrand. Das
Modell hat keine Spalte "Lage", also drückt es einen Teil der Lage über das Baujahr aus. Es
misst eine **Korrelation**, keine Kausalität: Ein Haus wird nicht wertvoller, wenn man es
altern lässt.

**2. Merkmale teilen sich die Erklärungskraft.** Im Modell mit einem Merkmal musste die
Wohnfläche allein alles erklären, auch den Teil, der eigentlich zur Ausstattung gehört. Jetzt
übernimmt `qualitaet` diesen Anteil — und `wohnflaeche_qm` bekommt ihren "ehrlicheren" Wert.

Genau deshalb ist ein Koeffizient nie eine schlichte Tatsache über die Welt, sondern immer eine
Aussage *innerhalb dieses Modells, mit diesen Merkmalen, auf diesen Daten*.
</details>

### 🛠️ Bonus-Challenge 2 — Wer trägt wie viel bei?

Wir haben zwei Merkmale gleichzeitig dazugenommen. Aber welches der beiden bringt eigentlich
wie viel?

Trainiere zwei weitere Modelle — eines mit `["wohnflaeche_qm", "qualitaet"]`, eines mit
`["wohnflaeche_qm", "baujahr"]` — und vergleiche ihr R² auf den Testdaten.

*Tipp: Die Spalten holst du dir genauso wie oben:
`train_multi[["wohnflaeche_qm", "qualitaet"]].to_numpy(dtype=float)`.*

In [ ]:
ergebnisse = {}

for merkmale in [["wohnflaeche_qm", "qualitaet"], ["wohnflaeche_qm", "baujahr"]]:
    # TODO 1: Trainings- und Testtabelle für genau diese Merkmale bauen
    X_tr = ...
    X_te = ...

    # TODO 2: Modell trainieren und R² auf den Testdaten berechnen
    modell_test = ...
    r2 = ...

    ergebnisse[" + ".join(merkmale)] = r2
    print(f"{' + '.join(merkmale):<32} R² = {r2:.4f}")

In [ ]:
# ✅ Selbsttest
assert abs(ergebnisse["wohnflaeche_qm + qualitaet"] - 0.5275) < 0.005
assert abs(ergebnisse["wohnflaeche_qm + baujahr"] - 0.5207) < 0.005
print()
print(f"{'nur Fläche':<32} R² = {r2_einfach:.4f}")
print(f"{'alle drei':<32} R² = {r2_multi:.4f}")
print("✅ Bonus-Challenge 2 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
ergebnisse = {}

for merkmale in [["wohnflaeche_qm", "qualitaet"], ["wohnflaeche_qm", "baujahr"]]:
    X_tr = train_multi[merkmale].to_numpy(dtype=float)
    X_te = test_multi[merkmale].to_numpy(dtype=float)

    modell_test = LinearRegression().fit(X_tr, y_train)
    r2 = r2_score(y_test, modell_test.predict(X_te))

    ergebnisse[" + ".join(merkmale)] = r2
    print(f"{' + '.join(merkmale):<32} R² = {r2:.4f}")
```

`train_multi[merkmale]` funktioniert mit einer *Liste* von Spaltennamen genauso wie mit einer
einzelnen — deshalb reicht dieselbe Zeile für beide Durchläufe der Schleife. Und `.fit(...)`
gibt das Modell zurück, lässt sich also direkt an `LinearRegression()` anhängen.

</details>

💬 **Was fällt auf?**

<details>
<summary>Antwort aufklappen</summary>

Jedes der beiden Merkmale bringt für sich genommen etwa **+0,03** (0,494 → 0,528 bzw. 0,521).
Zusammen bringen sie aber **+0,09** — deutlich mehr als die Summe der Einzelbeiträge.

Der Grund: Die beiden Merkmale erklären *unterschiedliche* Anteile des Preises und stören sich
kaum gegenseitig. Qualität steht für Ausstattung, Baujahr indirekt für die Lage. Erst zusammen
kann das Modell "altes Haus in guter Lage mit hochwertiger Ausstattung" von "Neubau am Stadtrand
mit Standardausstattung" unterscheiden.

Merkmale einzeln zu bewerten führt also leicht in die Irre — was zählt, ist ihr Zusammenspiel.
</details>

In [ ]:
# ▶️ Zum Spielen: eigene Häuser einsetzen und ausführen
haeuser = [
    [90.0, 6.0, 1960.0],
    [140.0, 8.0, 1995.0],
    [250.0, 10.0, 2010.0],
]

for haus, preis in zip(haeuser, modell_multi.predict(haeuser)):
    print(f"{haus[0]:>5.0f} m², Qualität {haus[1]:>2.0f}, Baujahr {haus[2]:.0f}  →  {preis:>10,.0f} USD")

---
## Was du gelernt hast

* Mehrere Merkmale bedeuten nur: **mehr Gewichte**. Die Formel bleibt dieselbe, der Code auch —
  statt einer Spalte übergibst du eine Tabelle.
* Merkmale hinzuzufügen ist der **wirksamste Hebel**, den du hast. Zwei zusätzliche Spalten haben
  hier mehr gebracht als jede Feinjustierung am Modell es könnte.
* Die gelernten Gewichte sind **interpretierbar**, aber keine Aussagen über die Welt. Sie gelten
  "alles andere gleich" — und sie füllen Lücken, wenn ein wichtiges Merkmal fehlt (hier: die Lage
  versteckt sich im Baujahr).

---
## Noch mehr?

1. **Die Lage dazunehmen.** Der Originaldatensatz auf Kaggle hat 21 Spalten, darunter `lat` und
   `long`. Lade ihn mit `kagglehub` und schau, wie weit du R² treiben kannst. (Achtung:
   Koordinaten wirken nicht linear auf den Preis — warum ist das ein Problem für unser Modell?)
2. **Ausreißer.** Entferne alle Häuser über 2 Mio. USD aus den *Trainingsdaten* und trainiere
   neu. Wird das Modell besser oder schlechter — und auf welchen Testdaten misst du das fair?
3. **Ein Merkmal, das keins ist.** Füge eine Spalte mit reinem Zufall hinzu
   (`np.random.default_rng(0).normal(size=len(X_train))`). Wie verändert sich R² auf den
   Trainingsdaten, wie auf den Testdaten? Was sagt dir das über R² als Maßstab?
4. **Der Gradientenabstieg dazu:** `bonus_gradientenabstieg_challenge.ipynb` zeigt, wie
   `scikit-learn` diese Gewichte überhaupt findet.